### [DELETE] Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.

This notebook will:
1. Purge soft-deleted ML Workspaces and API Management services (must be done BEFORE resource group deletion)
2. Delete all resources and purge remaining soft-deleted services (Key Vault, Cognitive Services, Log Analytics)
3. Delete the resource group
4. Sweep for any remaining soft-deleted resources that might block future redeployments

In [15]:
import os, sys, subprocess, json, time

def run_cmd(cmd, print_output=True):
    """Run Azure CLI command and return result"""
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    output = (proc.stdout or "").strip()
    error = (proc.stderr or "").strip()
    
    if print_output and output:
        print(output)
    if error and print_output:
        print(error)
    
    return {"success": proc.returncode == 0, "output": output, "error": error}

deployment_name = "azureml-integration-with-agents"
resource_group = f"lab-{deployment_name}"

print(f"[DELETE] Starting comprehensive cleanup for resource group: {resource_group}\n")

# Step 0: Purge soft-deleted ML Workspaces (must be done BEFORE resource group deletion)
print("[0/7] Checking for soft-deleted ML Workspaces...")
sub_info = run_cmd('az account show --query id -o tsv', print_output=False)
if sub_info['success']:
    subscription_id = sub_info['output'].strip()
    # Try REST API approach to purge soft-deleted workspaces
    ml_purge_cmd = f'az rest --method post --url "/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.MachineLearningServices/workspaces/aml-zmcdaeah6mdto?api-version=2024-04-01&forceToPurge=true" --headers "Content-Type=application/json" 2>/dev/null'
    ml_purge = run_cmd(ml_purge_cmd, print_output=False)
    if ml_purge['success'] or 'Not Found' in ml_purge['error']:
        print(f"  [OK] ML Workspace purge completed or no workspace found\")\n")
    else:
        print(f"  [INFO] ML Workspace cleanup attempted\\n")
else:
    print(f"  [WARN] Unable to get subscription ID\\n")

# Step 1: Purge soft-deleted APIM services (must be done BEFORE resource group deletion)
print("[1/7] Checking for soft-deleted API Management services...")
apim_list = run_cmd('az apim list-deleted --query "[].{name:name, location:location}" -o json', print_output=False)
if apim_list['success'] and apim_list['output']:
    try:
        deleted_apims = json.loads(apim_list['output'])
        if deleted_apims:
            for apim in deleted_apims:
                apim_name = apim.get('name')
                apim_location = apim.get('location')
                if apim_name:
                    print(f"  Purging API Management: {apim_name}")
                    purge_result = run_cmd(f'az apim purge --name {apim_name} --location {apim_location}', print_output=False)
                    if purge_result['success']:
                        print(f"    [OK] Purged: {apim_name}")
                    else:
                        print(f"    [WARN] Purge initiated (may take a moment): {apim_name}")
        else:
            print(f"  [OK] No soft-deleted API Management services found")
    except:
        print(f"  [OK] No soft-deleted API Management services found")
else:
    print(f"  [OK] No soft-deleted API Management services found")

print()

# Step 2: Delete the resource group
check = run_cmd(f'az group exists --name {resource_group}', print_output=False)
rg_exists = check['output'].strip() == 'true'

if rg_exists:
    print(f"[2/7] Deleting resource group: {resource_group}")
    result = run_cmd(f'az group delete --name {resource_group} --yes --no-wait')
    if result['success']:
        print(f"[OK] Resource group deletion initiated (running in background)\n")
    else:
        print(f"[FAILED] Failed to delete resource group: {result['error']}\n")
else:
    print(f"[2/7] Resource group '{resource_group}' already deleted or doesn't exist\n")

# Step 3: Purge soft-deleted Key Vaults
print("[3/7] Checking for soft-deleted Key Vaults...")
kv_list = run_cmd('az keyvault list-deleted --query "[].{name:name, location:properties.location}" -o json', print_output=False)
if kv_list['success'] and kv_list['output']:
    try:
        deleted_kvs = json.loads(kv_list['output'])
        if deleted_kvs:
            for kv in deleted_kvs:
                kv_name = kv.get('name')
                kv_location = kv.get('location')
                if kv_name:
                    print(f"  Purging Key Vault: {kv_name}")
                    purge_result = run_cmd(f'az keyvault purge --name {kv_name} --location {kv_location}', print_output=False)
                    if purge_result['success']:
                        print(f"    [OK] Purged: {kv_name}")
                    else:
                        print(f"    [WARN] Purge initiated (may take a moment): {kv_name}")
        else:
            print(f"  [OK] No soft-deleted Key Vaults found")
    except:
        print(f"  [OK] No soft-deleted Key Vaults found")
else:
    print(f"  [OK] No soft-deleted Key Vaults found")

print()

# Step 4: Purge soft-deleted Log Analytics workspaces
print("[4/7] Checking for soft-deleted Log Analytics workspaces...")
law_check = run_cmd(f'az monitor log-analytics workspace list --resource-group {resource_group} -o json', print_output=False)
if law_check['success'] and law_check['output']:
    try:
        deleted_laws = json.loads(law_check['output'])
        if deleted_laws:
            for law in deleted_laws:
                law_name = law.get('name')
                if law_name:
                    print(f"  Deleting Log Analytics workspace: {law_name}")
                    delete_result = run_cmd(f'az monitor log-analytics workspace delete --resource-group {resource_group} --workspace-name {law_name} --yes', print_output=False)
                    if delete_result['success']:
                        print(f"    [OK] Deleted: {law_name}")
                    else:
                        print(f"    [WARN] Delete initiated (may take a moment): {law_name}")
        else:
            print(f"  [OK] No soft-deleted Log Analytics workspaces found")
    except:
        print(f"  [OK] No soft-deleted Log Analytics workspaces found")
else:
    print(f"  [OK] No soft-deleted Log Analytics workspaces found")

print()

# Step 5: Purge soft-deleted Cognitive Services
print("[5/7] Checking for soft-deleted Cognitive Services...")
cog_list = run_cmd('az cognitiveservices account list-deleted --query "[].name" -o json', print_output=False)
if cog_list['success'] and cog_list['output']:
    try:
        deleted_cogs = json.loads(cog_list['output'])
        if deleted_cogs:
            for cog in deleted_cogs:
                print(f"  Purging Cognitive Service: {cog}")
                purge_result = run_cmd(f'az cognitiveservices account purge --name {cog} --resource-group {resource_group}', print_output=False)
                if purge_result['success']:
                    print(f"    [OK] Purged: {cog}")
                else:
                    print(f"    [WARN] Purge initiated (may take a moment): {cog}")
        else:
            print(f"  [OK] No soft-deleted Cognitive Services found")
    except:
        print(f"  [OK] No soft-deleted Cognitive Services found")
else:
    print(f"  [OK] No soft-deleted Cognitive Services found")

print(f"\n[OK] Cleanup process initiated. Soft-deleted resources may take 10-15 minutes to fully purge.")

[DELETE] Starting comprehensive cleanup for resource group: lab-azureml-integration-with-agents

[0/7] Checking for soft-deleted ML Workspaces...
  [INFO] ML Workspace cleanup attempted\n
[1/7] Checking for soft-deleted API Management services...
  [OK] No soft-deleted API Management services found

[2/7] Resource group 'lab-azureml-integration-with-agents' already deleted or doesn't exist

[3/7] Checking for soft-deleted Key Vaults...
  [OK] No soft-deleted Key Vaults found

[4/7] Checking for soft-deleted Log Analytics workspaces...
  [OK] No soft-deleted Log Analytics workspaces found

[5/7] Checking for soft-deleted Cognitive Services...
  Purging Cognitive Service: foundry1-33005-zmcdaeah6mdto
    [WARN] Purge initiated (may take a moment): foundry1-33005-zmcdaeah6mdto

[OK] Cleanup process initiated. Soft-deleted resources may take 10-15 minutes to fully purge.


### [SWEEP] Verify cleanup and check for remaining soft-deleted resources

If the resource group was already deleted (e.g., via the Azure Portal), soft-deleted resources may still linger and block redeployment with the same names. This cell sweeps for and verifies them.

In [16]:
# Verify final cleanup status
print(f"\n[STATUS] Verifying cleanup status...\n")

# Check if resource group still exists
final_check = run_cmd(f'az group exists --name {resource_group}', print_output=False)
rg_status = final_check['output'].strip().lower() == 'true'

if not rg_status:
    print(f"[OK] Resource group '{resource_group}' successfully deleted")
else:
    print(f"[WARN] Resource group still exists (deletion in progress)")

# Check for remaining soft-deleted resources
print(f"\n[RESOURCES] Final soft-deleted resources status:\n")

def _count_deleted(cmd: str) -> int:
    result = run_cmd(cmd, print_output=False)
    if not result['success']:
        return 0
    value = (result['output'] or '').strip()
    if not value:
        return 0
    try:
        return int(value)
    except Exception:
        try:
            parsed = json.loads(value)
            if isinstance(parsed, int):
                return parsed
            if isinstance(parsed, list):
                return len(parsed)
        except Exception:
            pass
    return 0

apim_count = _count_deleted('az apim list-deleted --query "length(@)" -o tsv')
if apim_count > 0:
    print(f"  [WARN] {apim_count} soft-deleted API Management service(s) remaining (purge in progress)")
else:
    print(f"  [OK] No soft-deleted API Management services")

kv_count = _count_deleted('az keyvault list-deleted --query "length(@)" -o tsv')
if kv_count > 0:
    print(f"  [WARN] {kv_count} soft-deleted Key Vault(s) remaining (purge in progress)")
else:
    print(f"  [OK] No soft-deleted Key Vaults")

cog_count = _count_deleted('az cognitiveservices account list-deleted --query "length(@)" -o tsv')
if cog_count > 0:
    print(f"  [WARN] {cog_count} soft-deleted Cognitive Services account(s) remaining (purge in progress)")
else:
    print(f"  [OK] No soft-deleted Cognitive Services accounts")

print(f"\n[OK] Cleanup verification complete")
print(f"[TIME] Note: Soft-deleted resources may take 10-15 minutes to fully purge")
print(f"[NEXT] If you see soft-deleted resources still remaining, wait a few minutes and re-run this cell.")


[STATUS] Verifying cleanup status...

[OK] Resource group 'lab-azureml-integration-with-agents' successfully deleted

[RESOURCES] Final soft-deleted resources status:

  [OK] No soft-deleted API Management services
  [OK] No soft-deleted Key Vaults
  [WARN] 1 soft-deleted Cognitive Services account(s) remaining (purge in progress)

[OK] Cleanup verification complete
[TIME] Note: Soft-deleted resources may take 10-15 minutes to fully purge
[NEXT] If you see soft-deleted resources still remaining, wait a few minutes and re-run this cell.
